In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from pathlib import Path
import json, glob, shutil, subprocess, os
import requests
import pandas as pd
import numpy as np
from datasets import load_dataset
SAVE_PATH_ADDRESS = '/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data/CovidET-Appraisals/repo'

SAVE_PATH = Path(SAVE_PATH_ADDRESS)

In [15]:
COVIDET_REPO = 'https://github.com/honglizhan/CovidET-Appraisals-Public.git'

subprocess.run(["git", "clone", COVIDET_REPO, str(SAVE_PATH)], check=True)

print("Downloaded repo to:", SAVE_PATH)

Downloaded repo to: /content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/data/CovidET-Appraisals/repo


In [17]:
all_files = []
for root, dirs, files in os.walk(SAVE_PATH):
    for f in files:
        path = Path(root) / f
        all_files.append(path)

print("Number of files:", len(all_files))
for p in all_files[:100]:
    print(p.relative_to(SAVE_PATH))

Number of files: 96
.gitattributes
README.md
.git/description
.git/config
.git/packed-refs
.git/HEAD
.git/index
.git/hooks/pre-merge-commit.sample
.git/hooks/pre-commit.sample
.git/hooks/pre-push.sample
.git/hooks/pre-applypatch.sample
.git/hooks/pre-rebase.sample
.git/hooks/applypatch-msg.sample
.git/hooks/pre-receive.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/update.sample
.git/hooks/push-to-checkout.sample
.git/hooks/commit-msg.sample
.git/hooks/prepare-commit-msg.sample
.git/hooks/post-update.sample
.git/info/exclude
.git/refs/heads/main
.git/refs/remotes/origin/HEAD
.git/objects/pack/pack-49992f360d8ebcac5786279627129e0cf8d48d64.pack
.git/objects/pack/pack-49992f360d8ebcac5786279627129e0cf8d48d64.idx
.git/logs/HEAD
.git/logs/refs/remotes/origin/HEAD
.git/logs/refs/heads/main
Human_EVAL/EVAL_analysis.ipynb
Human_EVAL/EVAL_inter-rator_agreement.ipynb
Human_EVAL/TURKERS-EVAL_data-intersection-ALL_6_models.csv
Human_EVAL/TURKERS-EVAL_data-intersection-TOP_3_models.csv
LLM_

In [19]:
candidate_files = []
for ext in ["*.csv", "*.tsv", "*.json", "*.jsonl", "*.xlsx", "*.pkl"]:
    candidate_files.extend(SAVE_PATH.rglob(ext))

print("Candidate data files:")
for p in candidate_files:
    print("-", p.relative_to(SAVE_PATH))

Candidate data files:
- Human_EVAL/TURKERS-EVAL_data-intersection-ALL_6_models.csv
- Human_EVAL/TURKERS-EVAL_data-intersection-TOP_3_models.csv
- data/CovidET-Appraisals.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-1.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-2.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-3.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-4.csv
- LLM_responses/Responses-Cleaned/alpaca-13B/alpaca-13B_seed-5.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-1.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-2.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-3.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-4.csv
- LLM_responses/Responses-Cleaned/alpaca-7B/alpaca-native_seed-5.csv
- LLM_responses/Responses-Cleaned/chatgpt/chatgpt_seed-1.csv
- LLM_responses/Responses-Cleaned/chatgpt/chatgpt_seed-2.csv
- LLM_responses/Response

In [20]:
def load_any_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    
    if suffix == ".csv":
        return pd.read_csv(path)
    elif suffix == ".tsv":
        return pd.read_csv(path, sep="\t")
    elif suffix == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)
    elif suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    elif suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    elif suffix == ".pkl":
        return pd.read_pickle(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

loaded_tables = {}

for p in candidate_files:
    try:
        df = load_any_table(p)
        loaded_tables[str(p)] = df
        print("\n" + "="*80)
        print("FILE:", p.relative_to(SAVE_PATH))
        print("SHAPE:", df.shape)
        print("COLUMNS:")
        print(list(df.columns))
        display(df.head(3))
    except Exception as e:
        print("Could not load:", p, "| Error:", repr(e))


FILE: Human_EVAL/TURKERS-EVAL_data-intersection-ALL_6_models.csv
SHAPE: (372, 13)
COLUMNS:
['HIT ID', 'Assignment ID', 'Worker ID', 'Reddit ID', 'Reddit Post', 'Dim', 'Model', 'Scale', 'Rationale', 'factuality', 'justifies', 'relevance', 'usefulness']


,HIT ID,Assignment ID,Worker ID,Reddit ID,Reddit Post,Dim,Model,Scale,Rationale,factuality,justifies,relevance,usefulness
0,36U4VBVNQOCO5B89BDL35JJI3B5RUV,3ZQIG0FLQFFVD6CP5ZAFCOP8DDOVW6,AKA8TN8H8DQ6T,o7hz16,My country is in a third wave. Its never endin...,dim4,alpaca-13B,0,Tell us why.,no,no,1,no
1,36U4VBVNQOCO5B89BDL35JJI3B5RUV,3YGXWBAF71GN5VN54C9RR4IS8DD4CY,A2MO3EE6D0P3KR,o7hz16,My country is in a third wave. Its never endin...,dim4,alpaca-13B,0,Tell us why.,no,no,1,no
2,3VP28W7DUNJ8OKXAQ5KH5MKE259FZD,3I33IC7ZWG1P92BKGKUCSUV6OWT2AV,AKA8TN8H8DQ6T,o7hz16,My country is in a third wave. Its never endin...,dim4,alpaca-7B,3,The narrator thought they were able to cope wi...,minor,yes,3,maybe



FILE: Human_EVAL/TURKERS-EVAL_data-intersection-TOP_3_models.csv
SHAPE: (390, 13)
COLUMNS:
['HIT ID', 'Assignment ID', 'Worker ID', 'Reddit ID', 'Reddit Post', 'Dim', 'Scale', 'Rationale', 'Model', 'factuality', 'justifies', 'relevance', 'usefulness']


,HIT ID,Assignment ID,Worker ID,Reddit ID,Reddit Post,Dim,Scale,Rationale,Model,factuality,justifies,relevance,usefulness
0,3QXFBUZ4ZKFXGLZVV2RR6F5B1RVUGH,3WI0P0II62R4BTVHFK0FJ3BKV6JRDI,A1KEA2Z47S3UPI,o92e6y,I've been living life pretty much normally sin...,dim1,5,The narrator thought they were responsible to ...,alpaca-7B,yes,yes,5,yes
1,3QXFBUZ4ZKFXGLZVV2RR6F5B1RVUGH,3MD9PLUKKJDMZW8LEV8V1BPDIRTZNQ,AKA8TN8H8DQ6T,o92e6y,I've been living life pretty much normally sin...,dim1,5,The narrator thought they were responsible to ...,alpaca-7B,yes,yes,5,yes
2,3TC2K6WK9G193UKBTK5X8JTLUHZ28R,3AMYWKA6YCL2T7OMD5BXJ2WN3Y96OV,A1KEA2Z47S3UPI,o92e6y,I've been living life pretty much normally sin...,dim1,1,The text does not address the question of the ...,chatgpt,no,no,5,no



FILE: data/CovidET-Appraisals.csv
SHAPE: (281, 53)
COLUMNS:
['HIT ID', 'Assignment ID', 'Worker ID', 'Reddit ID', 'Reddit Post', 'dim1', 'dim1_rationale', 'dim2', 'dim2_rationale', 'dim3', 'dim3_rationale', 'dim4', 'dim4_rationale', 'dim5', 'dim5_rationale', 'dim6', 'dim6_rationale', 'dim7', 'dim7_rationale', 'dim8', 'dim8_rationale', 'dim9', 'dim9_rationale', 'dim10', 'dim10_rationale', 'dim11', 'dim11_rationale', 'dim12', 'dim12_rationale', 'dim13', 'dim13_rationale', 'dim14', 'dim14_rationale', 'dim15', 'dim15_rationale', 'dim16', 'dim16_rationale', 'dim17', 'dim17_rationale', 'dim18', 'dim18_rationale', 'dim19', 'dim19_rationale', 'dim20', 'dim20_rationale', 'dim21', 'dim21_rationale', 'dim22', 'dim22_rationale', 'dim23', 'dim23_rationale', 'dim24', 'dim24_rationale']


...

,Reddit ID,dim1,dim1_rationale,dim2,dim2_rationale,dim3,dim3_rationale,dim4,dim4_rationale,dim5,dim5_rationale,dim6,dim6_rationale,dim7,dim7_rationale,dim8,dim8_rationale,dim9,dim9_rationale,dim10,dim10_rationale,dim11,dim11_rationale,dim12,dim12_rationale,dim13,dim13_rationale,dim14,dim14_rationale,dim15,dim15_rationale,dim16,dim16_rationale,dim17,dim17_rationale,dim18,dim18_rationale,dim19,dim19_rationale,dim20,dim20_rationale,dim21,dim21_rationale,dim22,dim22_rationale,dim23,dim23_rationale,dim24,dim24_rationale
0,o6lpwn,9</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,1</s>,1</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I dont even know how to speak of this grief. I...,1</s>,I dont even know how to speak of this grief. I...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I dont even know how to speak of this grief. I...,5</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,9</s>,1</s>,1</s>,9</s>,I feel so so powerless. So guilty I didnt repl...,I dont even know how to speak of this grief. I...,1</s>,5</s>,I feel so so powerless. So guilty I didnt repl...,5</s>,"I expected it to happen, but it didnt happen t..."
1,o77vmk,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,9</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...,9</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,1</s>,1</s>,5</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...
2,o78h66,9</s>,Completely responsible</s>,1</s>,1</s>,1</s>,1</s>,1</s>,1</s>,9</s>,I feel like everything he says is a repeat of ...,9</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,1</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,1</s>,1</s>,1</s>,I feel like everything he says is a repeat of ...,1</s>,I don't expect any advice or solution to this....,5</s>,I feel dead inside and doing anything requires...,5</s>,5</s>,5</s>,I feel like everything he says is a repeat of ...,5</s>,I feel like everything he says is a repeat of ...,1</s>,I don't expect any advice or solution to this....,5</s>,"I'm not sure if I'm a good person, but I'm try...",9</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,9</s>,I feel like everything he says is a repeat of ...,1</s>,I'm not sure if I'm in a challenging situation...,5</s>,I'm not sure if I'm being completely honest wi...,5</s>,I'm not surprised by this situation.</s>



FILE: LLM_responses/Responses-Raw/flan-t5-xxl/flan-t5-xxl_seed-4.csv
SHAPE: (241, 49)
COLUMNS:
['Reddit ID', 'dim1', 'dim1_rationale', 'dim2', 'dim2_rationale', 'dim3', 'dim3_rationale', 'dim4', 'dim4_rationale', 'dim5', 'dim5_rationale', 'dim6', 'dim6_rationale', 'dim7', 'dim7_rationale', 'dim8', 'dim8_rationale', 'dim9', 'dim9_rationale', 'dim10', 'dim10_rationale', 'dim11', 'dim11_rationale', 'dim12', 'dim12_rationale', 'dim13', 'dim13_rationale', 'dim14', 'dim14_rationale', 'dim15', 'dim15_rationale', 'dim16', 'dim16_rationale', 'dim17', 'dim17_rationale', 'dim18', 'dim18_rationale', 'dim19', 'dim19_rationale', 'dim20', 'dim20_rationale', 'dim21', 'dim21_rationale', 'dim22', 'dim22_rationale', 'dim23', 'dim23_rationale', 'dim24', 'dim24_rationale']


,Reddit ID,dim1,dim1_rationale,dim2,dim2_rationale,dim3,dim3_rationale,dim4,dim4_rationale,dim5,dim5_rationale,dim6,dim6_rationale,dim7,dim7_rationale,dim8,dim8_rationale,dim9,dim9_rationale,dim10,dim10_rationale,dim11,dim11_rationale,dim12,dim12_rationale,dim13,dim13_rationale,dim14,dim14_rationale,dim15,dim15_rationale,dim16,dim16_rationale,dim17,dim17_rationale,dim18,dim18_rationale,dim19,dim19_rationale,dim20,dim20_rationale,dim21,dim21_rationale,dim22,dim22_rationale,dim23,dim23_rationale,dim24,dim24_rationale
0,o6lpwn,9</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,1</s>,1</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I dont even know how to speak of this grief. I...,1</s>,1</s>,1</s>,I dont even know how to speak of this grief. I...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I dont even know how to speak of this grief. I...,5</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,9</s>,1</s>,1</s>,1</s>,I feel so so powerless. So guilty I didnt repl...,I dont even know how to speak of this grief. I...,1</s>,5</s>,I feel so so powerless. So guilty I didnt repl...,5</s>,"I expected it to happen, but it didnt happen t..."
1,o77vmk,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,9</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...,9</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...
2,o78h66,9</s>,Completely responsible</s>,1</s>,1</s>,1</s>,1</s>,1</s>,1</s>,9</s>,I feel like everything he says is a repeat of ...,9</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,1</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,1</s>,1</s>,1</s>,I feel like everything he says is a repeat of ...,1</s>,I'm not at all threatened.</s>,5</s>,I feel dead inside and doing anything requires...,5</s>,5</s>,5</s>,I don't expect any advice or solution to this....,5</s>,I feel dead inside and doing anything requires...,1</s>,I don't expect any advice or solution to this....,5</s>,I don't expect any advice or solution to this....,9</s>,"I feel like I'm a ghost, I feel like I'm a gho...",1</s>,1</s>,9</s>,I feel like everything he says is a repeat of ...,1</s>,I'm not sure if I'm a victim of the pandemic o...,5</s>,"I'm not sure if I'm being overly dramatic, but...",5</s>,I feel like everything he says is a repeat of ...



FILE: LLM_responses/Responses-Raw/flan-t5-xxl/flan-t5-xxl_seed-5.csv
SHAPE: (241, 49)
COLUMNS:
['Reddit ID', 'dim1', 'dim1_rationale', 'dim2', 'dim2_rationale', 'dim3', 'dim3_rationale', 'dim4', 'dim4_rationale', 'dim5', 'dim5_rationale', 'dim6', 'dim6_rationale', 'dim7', 'dim7_rationale', 'dim8', 'dim8_rationale', 'dim9', 'dim9_rationale', 'dim10', 'dim10_rationale', 'dim11', 'dim11_rationale', 'dim12', 'dim12_rationale', 'dim13', 'dim13_rationale', 'dim14', 'dim14_rationale', 'dim15', 'dim15_rationale', 'dim16', 'dim16_rationale', 'dim17', 'dim17_rationale', 'dim18', 'dim18_rationale', 'dim19', 'dim19_rationale', 'dim20', 'dim20_rationale', 'dim21', 'dim21_rationale', 'dim22', 'dim22_rationale', 'dim23', 'dim23_rationale', 'dim24', 'dim24_rationale']


,Reddit ID,dim1,dim1_rationale,dim2,dim2_rationale,dim3,dim3_rationale,dim4,dim4_rationale,dim5,dim5_rationale,dim6,dim6_rationale,dim7,dim7_rationale,dim8,dim8_rationale,dim9,dim9_rationale,dim10,dim10_rationale,dim11,dim11_rationale,dim12,dim12_rationale,dim13,dim13_rationale,dim14,dim14_rationale,dim15,dim15_rationale,dim16,dim16_rationale,dim17,dim17_rationale,dim18,dim18_rationale,dim19,dim19_rationale,dim20,dim20_rationale,dim21,dim21_rationale,dim22,dim22_rationale,dim23,dim23_rationale,dim24,dim24_rationale
0,o6lpwn,9</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,1</s>,1</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,I feel so so powerless. So guilty I didnt repl...,9</s>,1</s>,I dont even know how to speak of this grief. I...,1</s>,I dont even know how to speak of this grief. I...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I feel so so powerless. So guilty I didnt repl...,1</s>,I dont even know how to speak of this grief. I...,5</s>,I feel so so powerless. So guilty I didnt repl...,9</s>,9</s>,1</s>,1</s>,1</s>,I dont even know how to speak of this grief. I...,I dont even know how to speak of this grief. I...,1</s>,5</s>,I feel so so powerless. So guilty I didnt repl...,5</s>,"I expected it to happen, but it didnt happen t..."
1,o77vmk,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,9</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,I'm being vague as to not give away my employe...,1</s>,9</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,1</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...,9</s>,I'm being vague as to not give away my employe...,1</s>,1</s>,1</s>,1</s>,1</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...,5</s>,I'm being vague as to not give away my employe...
2,o78h66,9</s>,Completely responsible</s>,1</s>,1</s>,1</s>,1</s>,1</s>,1</s>,9</s>,I feel like everything he says is a repeat of ...,9</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,1</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,1</s>,1</s>,1</s>,I feel like everything he says is a repeat of ...,1</s>,I'm not at all threatened.</s>,5</s>,I'm not sure if I'm in a better or worse place...,5</s>,5</s>,5</s>,I feel like everything he says is a repeat of ...,5</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,5</s>,I don't expect any advice or solution to this....,9</s>,I feel like everything he says is a repeat of ...,1</s>,1</s>,9</s>,I feel like everything he says is a repeat of ...,1</s>,I feel dead inside and doing anything requires...,5</s>,I'm not sure if I'm a person who values person...,5</s>,5</s>


In [22]:
import pandas as pd

covid_df = pd.read_csv(SAVE_PATH / 'data/CovidET-Appraisals.csv')

print("Shape:", covid_df.shape)
print("Unique Reddit posts:", covid_df["Reddit ID"].nunique())
print("Unique annotators:", covid_df["Worker ID"].nunique())
print(covid_df["Worker ID"].value_counts())

rating_cols = [c for c in covid_df.columns if c.startswith("dim") and not c.endswith("_rationale")]
rationale_cols = [c for c in covid_df.columns if c.endswith("_rationale")]

print("Rating columns:", len(rating_cols))
print("Rationale columns:", len(rationale_cols))

all_ratings = pd.concat([covid_df[c] for c in rating_cols])
print("Rating min:", all_ratings.dropna().min())
print("Rating max:", all_ratings.dropna().max())
print("Unique rating values:", sorted(all_ratings.dropna().unique()))
print("Missing ratings:", all_ratings.isna().sum())

Shape: (281, 53)
Unique Reddit posts: 241
Unique annotators: 2
Worker ID
A23ARVB31O6LE3    143
AM6H40LNWSFYA     138
Name: count, dtype: int64
Rating columns: 24
Rationale columns: 24
Rating min: 1.0
Rating max: 9.0
Unique rating values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0)]
Missing ratings: 1052


In [23]:
covid_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 281 entries, 0 to 280
Data columns (total 53 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   HIT ID           281 non-null    object 
 1   Assignment ID    281 non-null    object 
 2   Worker ID        281 non-null    object 
 3   Reddit ID        281 non-null    object 
 4   Reddit Post      281 non-null    object 
 5   dim1             277 non-null    float64
 6   dim1_rationale   281 non-null    object 
 7   dim2             266 non-null    float64
 8   dim2_rationale   281 non-null    object 
 9   dim3             273 non-null    float64
 10  dim3_rationale   281 non-null    object 
 11  dim4             278 non-null    float64
 12  dim4_rationale   281 non-null    object 
 13  dim5             260 non-null    float64
 14  dim5_rationale   281 non-null    object 
 15  dim6             271 non-null    float64
 16  dim6_rationale   281 non-null    object 
 17  dim7            

In [24]:
covid_df.describe()

,dim1,dim2,dim3,dim4,dim5,dim6,dim7,dim8,dim9,dim10,dim11,dim12,dim13,dim14,dim15,dim16,dim17,dim18,dim19,dim20,dim21,dim22,dim23,dim24
count,277.000000,266.000000,273.000000,278.000000,260.000000,271.000000,261.000000,273.000000,260.000000,276.000000,218.000000,272.000000,275.000000,277.000000,268.000000,44.000000,238.000000,69.000000,270.000000,257.000000,249.000000,268.000000,41.000000,251.000000
mean,2.808664,5.259398,7.249084,3.964029,7.926923,6.391144,3.739464,3.516484,4.973077,7.155797,4.504587,5.404412,2.705455,4.808664,2.570896,2.681818,4.260504,3.521739,4.540741,3.844358,6.698795,7.220149,2.634146,3.338645
std,2.077187,2.780730,1.871872,2.139233,0.969838,2.169262,2.246607,2.398051,2.751491,2.055857,2.296557,2.599684,1.581302,2.438266,1.704426,1.325470,1.625027,1.441085,2.903050,2.223608,1.573994,1.540965,1.996949,2.051551
min,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,2.000000,7.000000,3.000000,7.000000,6.000000,2.000000,1.000000,2.000000,7.000000,3.000000,4.000000,1.000000,3.000000,1.000000,1.000000,3.000000,3.000000,1.000000,1.000000,6.000000,6.000000,1.000000,1.000000
50%,2.000000,6.000000,8.000000,3.000000,8.000000,7.000000,3.000000,3.000000,6.000000,8.000000,4.000000,6.000000,3.000000,4.000000,2.000000,3.000000,4.000000,4.000000,6.000000,4.000000,7.000000,7.000000,2.000000,3.000000
75%,4.000000,7.000000,9.000000,6.000000,9.000000,8.000000,4.000000,6.000000,7.000000,8.000000,7.000000,7.000000,4.000000,7.000000,3.000000,4.000000,5.000000,4.000000,7.000000,6.000000,8.000000,8.000000,4.000000,5.000000
max,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,5.000000,9.000000,7.000000,9.000000,9.000000,9.000000,9.000000,9.000000,8.000000


In [25]:
covid_df.head()

,HIT ID,Assignment ID,Worker ID,Reddit ID,Reddit Post,dim1,dim1_rationale,dim2,dim2_rationale,dim3,dim3_rationale,dim4,dim4_rationale,dim5,dim5_rationale,dim6,dim6_rationale,dim7,dim7_rationale,dim8,dim8_rationale,dim9,dim9_rationale,dim10,dim10_rationale,dim11,dim11_rationale,dim12,dim12_rationale,dim13,dim13_rationale,dim14,dim14_rationale,dim15,dim15_rationale,dim16,dim16_rationale,dim17,dim17_rationale,dim18,dim18_rationale,dim19,dim19_rationale,dim20,dim20_rationale,dim21,dim21_rationale,dim22,dim22_rationale,dim23,dim23_rationale,dim24,dim24_rationale
0,360ZO6N6KAUVP7FXL2X2G8VG6P9M9W,3QEMNNSB27AQUEJLJE5G86CWSFZ7D6,AM6H40LNWSFYA,o6lpwn,I dont even know how to speak of this grief. I...,6.0,While the narrator doesn't seem to feel respon...,2.0,The narrator doesn't believe that others in th...,7.0,The narrator seems to think that the cause of ...,2.0,The narrator doesn't seem to believe that they...,8.0,The narrator believes that their Uncle's death...,3.0,Although the narrator mentions having to learn...,2.0,The narrator does not seem to think that they ...,3.0,The narrator expresses regret not picking up a...,2.0,The narrator doesn't seem to think that other ...,8.0,The narrator seems to think that their uncle's...,2.0,The narrator expresses an inability to know ho...,NaN,The narrator doesn't mention any particular th...,1.0,The narrator does not think that their uncle's...,3.0,The narrator expresses a lot of uncertainty in...,1.0,The narrator thinks that their uncle's death w...,2.0,The narrator does not think that their uncle's...,2.0,"The narrator thinks that their situation, afte...",NaN,The narrator doesn't really mention any extern...,9.0,The narrator thinks that something has been co...,2.0,The narrator does not think that their uncle's...,7.0,The narrator expresses that their uncle's deat...,8.0,The narrator thinks that their uncle's death i...,NaN,The narrator doesn't really mention any person...,2.0,The narrator did not expect their uncle to die...
1,360ZO6N6KAUVP7FXL2X2G8VG6P9M9W,3PM8NZGV88REY2TH6DFY24WJDNCXQ5,A23ARVB31O6LE3,o6lpwn,I dont even know how to speak of this grief. I...,2.0,While the narrator understands they did not pl...,2.0,The narrator mentions how they were unable to ...,9.0,The narrator brings up COVID as the primary ca...,1.0,The narrator mentions feelings of extreme guil...,9.0,The narrator states how they and their uncle w...,5.0,The narrator expresses an understanding of how...,1.0,The narrator expresses a complete hopelessness...,1.0,The narrator states that they feel 'powerless'...,1.0,The narrator states that their uncle took as m...,9.0,The narrator seems to believe that their uncle...,NaN,The narrator does not mention whether or not t...,6.0,The narrator seems to perceive COVID and the p...,1.0,The narrator does not find the situation pleas...,4.0,"While the death is recent, the narrator unders...",1.0,The situation is completely inconsistent with ...,1.0,The narrator does not find the situation fair ...,4.0,While the narrator does not state whether they...,NaN,The narrator does not express how they feel th...,9.0,The narrator has experienced the death of a ve...,1.0,The narrator does not mention ever having face...,9.0,The narrator's uncle's death is taking a huge ...,9.0,The narrator is facing extreme challenge and h...,1.0,The narrator seems to have a strong care for t...,1.0,Though they narrator understands the risk and ...
2,3Z33IC0JD9XEFU96ZUYT34ETXILV9M,33IZTU6J8BCQBI8UYH43X4BKVBSSXK,A23ARVB31O6LE3,o77vmk,I'm being vague as to not give away my employe...,1.0,The narrator only mentions other people or for...,9.0,The narrator mentions how their employer is no...,7.0,The narrator acknowledges that COVID has cause...,1.0,The narrator explicitly expresses frustration ...,8.0,"The narrator wants to unmask, just as their fa...",6.0,The narrator seems to think the situation is c...,3.0,The narrator does not seem anxious or worried ...,2.0,The narrator mentions how they have been vacci...